Code to set path root

In [ ]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 32x32 Image Sizes
* No ResNet
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

### Adding 99 blocks (MaxPool2D in each second block)

In [ ]:
in_channels = 3
out_channels = 4
size = 32

model_blocks = []

for i in range(1,100):
    conv = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
    batch_norm = nn.BatchNorm2d(out_channels)
    model_blocks.extend(
        [conv, batch_norm, nn.ReLU()]
    )
    if i%30==0:
        model_blocks.append(nn.MaxPool2d(2,2))
        size = size//2
    
    in_channels = out_channels
    if i%10 == 0:out_channels = out_channels * 2

print(f"Final In Channels = {in_channels}")
print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

In [ ]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(in_channels * size * size, in_channels),
    nn.ReLU(),
    nn.Linear(in_channels, 2048),
    nn.ReLU(),
    nn.Linear(2048, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
    )

### Use Trainer to train and check validations
Adding weight decay and decreased weight

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment5/",
    save_checkpoints=10,
    print_every=5
    )

In [ ]:
history = trainer.fit(100)

### Save Metrics

In [ ]:
df = plot_training_metrics(history)
df.to_csv("../documentations/experiments/experiment5/tables/training_metrics.csv", index=False)

### Training/Validation Trend (100 epochs)
* No real convergence: Training and validation loss remain almost flat around ~1.08–1.11 across 100 epochs, indicating the model is not learning meaningful separations.
* Performance near random baseline: Train/validation accuracy stays mostly in the 0.33–0.37 range, which is close to random guessing for a 3-class problem (~0.33).
* Weak discriminative power: F1 scores remain low (~0.20–0.30), showing poor balance between precision and recall.
* Severe prediction collapse: Confusion matrices repeatedly show the model heavily biased toward a single class (often the middle column), suggesting mode collapse / majority-class prediction behavior.
* No clear overfitting: Train and validation metrics are similarly low and move together, indicating underfitting rather than overfitting.
* High instability in validation precision/recall: Some epochs show sudden spikes (e.g., precision >0.5–0.6) but without sustained improvement, indicating unstable optimization rather than real learning.
* Optimization stagnation: After early epochs, improvements plateau completely, suggesting issues like learning rate or model capacity.

The training process shows a model that fails to learn meaningful structure from the data. Across 100 epochs, both training and validation losses remain nearly constant around 1.09, and accuracy stays close to chance level for a 3-class classification task. The F1 scores remain low and inconsistent, while confusion matrices reveal a strong bias toward predicting a single dominant class. This indicates a collapse in decision diversity rather than proper class separation. Since training and validation performance are similarly poor, the issue is not overfitting but underfitting or optimization failure. Overall, the model appears stuck in a non-improving regime, likely due to limited representational capacity, suboptimal training setup.

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
import copy

test_scores = []

# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

# Loops for testing
for i in range(11):
    test_model = copy.deepcopy(model)

    checkpoint = torch.load(
        f"../models/experiment5/epoch_{i*10}.pt" if i>0 else "../models/experiment5/best_model.pt",
        map_location="cuda"
    )

    test_model.load_state_dict(checkpoint["model"])

    tester = Tester(
        test_model,
        test_loader,
        3,
        torch.nn.CrossEntropyLoss(),
        "cuda"
    )

    result = tester.test(return_predictions=True)

    test_scores.append(result)


### Save Test Metrics

In [ ]:
# Plot all 100 epochs
test_metrics_df = plot_testing_history(test_scores)
test_metrics_df.to_csv("../documentations/experiments/experiment5/tables/test_metrics.csv", index=False)

### Test Performance Trend (100 epochs)
* Stable but low performance: Test accuracy remains in the range 0.34–0.37, closely matching validation performance and staying near chance level for a 3-class problem.
* No generalization gap: Test metrics are very similar to training/validation trends, confirming the model is not overfitting, but instead underfitting.
* Loss instability: Test loss fluctuates between ~1.09 and 1.14, without a downward trend, indicating inconsistent predictive confidence.
* Weak F1 performance: Test F1 stays low (~0.18–0.29), showing poor balance between precision and recall across classes.
* High variance in precision/recall: Precision and recall fluctuate significantly across epochs without stabilization, suggesting unstable class-wise predictions.
* Model collapse behavior persists: Consistent with training logs, predictions likely remain biased toward one or two dominant classes rather than learning class separation.

Across training, validation, and test evaluations, the model shows a consistent pattern of non-convergent learning. Loss values remain near the initial level (~1.09), and accuracy across all splits stays close to random chance, indicating the model is not extracting meaningful features from the input data. Both validation and test results mirror training behavior closely, confirming that the issue is not overfitting but systematic underfitting or optimization failure. The low and unstable F1 scores, along with weak precision-recall balance, suggest that the model collapses toward biased class predictions rather than learning discriminative boundaries. Overall, the system is not learning effectively and likely requires changes in architecture, data representation, or training configuration.